# Recipe Flowchart -- dev notebook


Mirrors `src/build_site.py` step by step, for exploring the pipeline interactively: load a recipe, extract structured steps with one model, generate the Mermaid flowchart, then compare all four models on cost/latency/structure.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "src")

from dotenv import load_dotenv
load_dotenv()

from recipe_parser import parse_recipe
from flowchart import to_mermaid
from build_site import RECIPES, RUNS

## 1. Load a recipe

The complex example is a photo of my own handwritten notes -- bilingual (Korean/English) shorthand with hand-drawn brackets already showing which sub-steps run in parallel. Good stress test for the vision-extraction path.

In [ ]:
recipe = RECIPES[2]  # the handwritten Choux au Craquelin photo
recipe["input"]

## 2. Extract structured steps

One forced tool-use call returns a title plus a list of steps, each with an id, imperative text, an estimated duration, and `depends_on` -- the field that encodes the dependency graph.

In [ ]:
result = parse_recipe(recipe["input"], provider="anthropic", model="claude-sonnet-5")

print(result.title)
for step in result.steps:
    print(f"  {step['id']:>3}  {step['text']:<55}  depends_on={step['depends_on']}  ({step['duration_min']} min)")

## 3. Generate the flowchart

`to_mermaid` walks the `depends_on` graph and emits Mermaid `graph TD` syntax -- independent steps render as parallel branches that converge wherever a later step depends on both.

In [ ]:
print(to_mermaid(result.steps))

Paste the output above into [mermaid.live](https://mermaid.live) to preview it directly -- the notebook doesn't render Mermaid natively.

## 4. Compare models

The real question: does a $1/MTok model get the *dependency graph* right, not just the step list? Run the same recipe through all four models and compare cost, latency, and structure (step count vs. how many steps it flagged as independent -- a proxy for whether it caught the parallelism).

In [ ]:
for provider, model, label in RUNS:
    r = parse_recipe(recipe["input"], provider=provider, model=model)
    n_independent = sum(1 for s in r.steps if not s["depends_on"])
    print(f"{label:<16}  {r.input_tokens:>5}in {r.output_tokens:>4}out  "
          f"${r.estimated_cost_usd:.4f}  {r.latency_s:5.1f}s  "
          f"{len(r.steps):>2} steps ({n_independent} independent)")

## 5. Build the full site

Runs all three recipes x all four models and regenerates `docs/index.html`.

In [ ]:
import build_site
build_site.main()